<a href="https://colab.research.google.com/github/jiangli001/wildfire-air-quality/blob/main/wildfire_pm25_training_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""wildfire_pm25_training_transformer_v2.ipynb

Improved Transformer-based PM2.5 Prediction Model
Features:
- Encoder-Decoder architecture (two-layer transformer)
- Learnable positional embeddings
- Multi-head attention with causal masking for decoder
- Skip connections and layer normalization
- Warmup learning rate schedule

Original file is located at
    https://colab.research.google.com/drive/1Z8HmenZRan6rGBBL1uFDtXoSVBx84ig-
"""

from google.colab import drive
drive.mount('/content/drive')

# Commented out IPython magic to ensure Python compatibility.
# %load_ext cudf.pandas

import pandas as pd
import numpy as np
from tqdm.auto import tqdm


def create_multivariate_windows(
    df,
    window_size=24,
    forecast_horizon=24,
    feature_cols=None,
    stride=1,
):
    """
    Create 2D sliding windows for multivariate time series modeling.
    Optimized for performance using NumPy operations.

    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with columns: site, date, start_hour, and feature columns
    window_size : int
        Size of the sliding window (default: 24 for hours in a day)
    forecast_horizon : int
        Number of time steps to predict ahead (default: 2)
    feature_cols : list
        List of column names to include as features
    stride : int
        Step size for sliding the window (default: 1)

    Returns:
    --------
    X : np.ndarray
        Array of shape (n_windows, window_size, n_features) containing input sequences
    y : np.ndarray
        Array of shape (n_windows, forecast_horizon, n_features) containing target sequences
    metadata : pd.DataFrame
        Dataframe containing metadata for each window
    """

    if feature_cols is None:
        feature_cols = []
    # Verify all feature columns exist
    missing_cols = [col for col in feature_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in dataframe: {missing_cols}")

    # Sort data by site, date, and start_hour
    df = df.sort_values(["site", "date", "start_hour"]).reset_index(drop=True)

    X = []
    y = []
    metadata = []

    # Process each site separately to ensure no cross-site windows
    unique_sites = df["site"].unique()
    print(f"Processing {len(unique_sites)} sites to create sliding windows...")

    for site in tqdm(unique_sites, desc="Sites"):
        site_data = df[df["site"] == site].reset_index(drop=True)

        # Convert to numpy arrays for fast slicing (Major Optimization)
        data_values = site_data[feature_cols].values
        dates = site_data["date"].values
        hours = site_data["start_hour"].values

        # Calculate the maximum valid starting index
        # We need window_size + forecast_horizon rows to create a valid sample
        max_start_idx = len(site_data) - window_size - forecast_horizon + 1

        if max_start_idx <= 0:
            continue

        # Create sliding windows with specified stride
        for i in range(0, max_start_idx, stride):
            # Indices
            window_end_idx = i + window_size
            target_start_idx = window_end_idx
            target_end_idx = target_start_idx + forecast_horizon

            # Extract windows using numpy slicing (Fast)
            X_window = data_values[i:window_end_idx]
            y_window = data_values[target_start_idx:target_end_idx]

            X.append(X_window)
            y.append(y_window)

            # Store metadata (using numpy array access is faster than .iloc)
            metadata.append(
                {
                    "site": site,
                    "window_start_idx": i,
                    "X_start_date": dates[i],
                    "X_start_hour": hours[i],
                    "X_end_date": dates[window_end_idx - 1],
                    "X_end_hour": hours[window_end_idx - 1],
                    "y_start_date": dates[target_start_idx],
                    "y_start_hour": hours[target_start_idx],
                    "y_end_date": dates[target_end_idx - 1],
                    "y_end_hour": hours[target_end_idx - 1],
                }
            )

    # Convert to numpy arrays
    X = np.array(X)  # Shape: (n_samples, window_size, n_features)
    y = np.array(y)  # Shape: (n_samples, forecast_horizon, n_features)
    metadata_df = pd.DataFrame(metadata)

    return X, y, metadata_df


def split_train_test_by_site(X, y, metadata, test_size=0.2, random_state=42):
    """
    Split data into train and test sets, keeping all windows from each site together.

    Parameters:
    -----------
    X : np.ndarray
        Input windows
    y : np.ndarray
        Target windows
    metadata : pd.DataFrame
        Metadata with site information
    test_size : float
        Proportion of sites to use for testing
    random_state : int
        Random seed for reproducibility

    Returns:
    --------
    X_train, X_test, y_train, y_test, metadata_train, metadata_test
    """
    np.random.seed(random_state)

    # Get unique sites
    unique_sites = metadata["site"].unique()
    n_test_sites = max(1, int(len(unique_sites) * test_size))

    # Randomly select test sites
    test_sites = np.random.choice(unique_sites, size=n_test_sites, replace=False)

    # Create train/test masks
    test_mask = metadata["site"].isin(test_sites)
    train_mask = ~test_mask

    return (
        X[train_mask],
        X[test_mask],
        y[train_mask],
        y[test_mask],
        metadata[train_mask].reset_index(drop=True),
        metadata[test_mask].reset_index(drop=True),
    )

import torch

# Check if CUDA is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using GPU.")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

print(f"Device set to: {device}")

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from pathlib import Path
import json
from datetime import datetime
import math

class PM25Dataset(Dataset):
    """PyTorch Dataset for PM2.5 time series data."""

    def __init__(self, X, y):
        """
        Args:
            X: np.ndarray of shape (n_samples, window_size, n_features)
            y: np.ndarray of shape (n_samples, forecast_horizon, n_features)
        """
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LearnablePositionalEncoding(nn.Module):
    """
    Learnable positional encoding - often works better than sinusoidal for
    fixed-length sequences like ours.
    """

    def __init__(self, d_model, max_len=100, dropout=0.1):
        super(LearnablePositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

    def forward(self, x):
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
        Returns:
            Tensor with positional encoding added
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TemporalEmbedding(nn.Module):
    """
    Enhanced input embedding with temporal features.
    Projects input features and adds learnable temporal embeddings.
    """

    def __init__(self, input_size, d_model, max_len=100, dropout=0.1):
        super(TemporalEmbedding, self).__init__()

        # Project input features to d_model
        self.input_projection = nn.Linear(input_size, d_model)

        # Learnable positional encoding
        self.pos_encoding = LearnablePositionalEncoding(d_model, max_len, dropout)

        # Layer norm after embedding
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len, input_size)
        Returns:
            (batch_size, seq_len, d_model)
        """
        x = self.input_projection(x)
        x = self.pos_encoding(x)
        x = self.layer_norm(x)
        return x


class PM25EncoderDecoderTransformer(nn.Module):
    """
    Two-Layer Transformer with Encoder-Decoder Architecture for PM2.5 prediction.

    Architecture:
    - Layer 1 (Encoder): Processes the input sequence (historical data)
    - Layer 2 (Decoder): Generates the output sequence (forecasts) using cross-attention

    This is more suitable for sequence-to-sequence prediction than encoder-only.
    """

    def __init__(
        self,
        input_size=1,
        d_model=128,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=512,
        dropout=0.1,
        forecast_horizon=24,
        output_size=1,
        window_size=24
    ):
        super(PM25EncoderDecoderTransformer, self).__init__()

        self.d_model = d_model
        self.forecast_horizon = forecast_horizon
        self.output_size = output_size
        self.window_size = window_size

        # ==================== LAYER 1: ENCODER ====================
        # Input embedding for encoder (historical sequence)
        self.encoder_embedding = TemporalEmbedding(
            input_size, d_model, max_len=window_size, dropout=dropout
        )

        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True  # Pre-norm architecture (more stable training)
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers,
            norm=nn.LayerNorm(d_model)
        )

        # ==================== LAYER 2: DECODER ====================
        # Decoder input: learnable query tokens for each forecast step
        self.decoder_query = nn.Parameter(torch.randn(1, forecast_horizon, d_model) * 0.02)

        # Positional encoding for decoder queries
        self.decoder_pos_encoding = LearnablePositionalEncoding(
            d_model, max_len=forecast_horizon, dropout=dropout
        )

        # Transformer decoder layers with cross-attention
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_decoder_layers,
            norm=nn.LayerNorm(d_model)
        )

        # ==================== OUTPUT PROJECTION ====================
        # Project decoder output to predictions
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, output_size)
        )

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize weights using Xavier initialization."""
        for name, p in self.named_parameters():
            if 'weight' in name and p.dim() > 1:
                nn.init.xavier_uniform_(p)
            elif 'bias' in name:
                nn.init.zeros_(p)

    def _generate_causal_mask(self, size, device):
        """Generate causal mask for decoder self-attention."""
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask

    def forward(self, x):
        """
        Forward pass.

        Args:
            x: Input tensor of shape (batch_size, window_size, input_size)

        Returns:
            Output tensor of shape (batch_size, forecast_horizon, output_size)
        """
        batch_size = x.size(0)
        device = x.device

        # ==================== ENCODER (Layer 1) ====================
        # Embed input sequence
        encoder_input = self.encoder_embedding(x)  # (batch, window_size, d_model)

        # Encode historical sequence
        memory = self.encoder(encoder_input)  # (batch, window_size, d_model)

        # ==================== DECODER (Layer 2) ====================
        # Expand decoder queries for batch
        decoder_input = self.decoder_query.expand(batch_size, -1, -1)  # (batch, forecast_horizon, d_model)
        decoder_input = self.decoder_pos_encoding(decoder_input)

        # Generate causal mask for autoregressive decoding
        tgt_mask = self._generate_causal_mask(self.forecast_horizon, device)

        # Decode with cross-attention to encoder output
        decoder_output = self.decoder(
            tgt=decoder_input,
            memory=memory,
            tgt_mask=tgt_mask
        )  # (batch, forecast_horizon, d_model)

        # ==================== OUTPUT ====================
        # Project to output size
        output = self.output_projection(decoder_output)  # (batch, forecast_horizon, output_size)

        return output


class PM25TransformerWithSkipConnections(nn.Module):
    """
    Alternative: Two-Layer Transformer with Skip Connections.

    This version uses two stacked transformer encoder blocks with skip connections
    between them, which can help with gradient flow.
    """

    def __init__(
        self,
        input_size=1,
        d_model=128,
        nhead=8,
        num_layers_per_block=2,
        dim_feedforward=512,
        dropout=0.1,
        forecast_horizon=24,
        output_size=1,
        window_size=24
    ):
        super(PM25TransformerWithSkipConnections, self).__init__()

        self.d_model = d_model
        self.forecast_horizon = forecast_horizon
        self.output_size = output_size

        # Input embedding
        self.embedding = TemporalEmbedding(
            input_size, d_model, max_len=window_size, dropout=dropout
        )

        # ==================== LAYER 1: First Transformer Block ====================
        encoder_layer_1 = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True
        )
        self.transformer_block_1 = nn.TransformerEncoder(
            encoder_layer_1,
            num_layers=num_layers_per_block,
            norm=nn.LayerNorm(d_model)
        )

        # Intermediate projection (for skip connection dimension matching if needed)
        self.intermediate_norm = nn.LayerNorm(d_model)

        # ==================== LAYER 2: Second Transformer Block ====================
        encoder_layer_2 = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True
        )
        self.transformer_block_2 = nn.TransformerEncoder(
            encoder_layer_2,
            num_layers=num_layers_per_block,
            norm=nn.LayerNorm(d_model)
        )

        # ==================== OUTPUT ====================
        # Global attention pooling
        self.attention_pool = nn.Sequential(
            nn.Linear(d_model, 1),
            nn.Softmax(dim=1)
        )

        # Output projection
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, forecast_horizon * output_size)
        )

        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if 'weight' in name and p.dim() > 1:
                nn.init.xavier_uniform_(p)
            elif 'bias' in name:
                nn.init.zeros_(p)

    def forward(self, x):
        batch_size = x.size(0)

        # Embed input
        x = self.embedding(x)  # (batch, seq_len, d_model)

        # Layer 1
        out_1 = self.transformer_block_1(x)  # (batch, seq_len, d_model)

        # Skip connection + normalization
        out_1 = self.intermediate_norm(out_1 + x)

        # Layer 2
        out_2 = self.transformer_block_2(out_1)  # (batch, seq_len, d_model)

        # Skip connection from layer 1
        out_2 = out_2 + out_1

        # Attention pooling
        attn_weights = self.attention_pool(out_2)  # (batch, seq_len, 1)
        pooled = (out_2 * attn_weights).sum(dim=1)  # (batch, d_model)

        # Project to output
        output = self.output_projection(pooled)  # (batch, forecast_horizon * output_size)
        output = output.view(batch_size, self.forecast_horizon, self.output_size)

        return output


class EarlyStopping:
    """Early stopping to prevent overfitting."""

    def __init__(self, patience=10, min_delta=0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

        return self.early_stop


class WarmupCosineScheduler:
    """
    Learning rate scheduler with warmup and cosine annealing.
    Warmup helps transformer training stability.
    """

    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        self.base_lr = optimizer.param_groups[0]['lr']

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            # Linear warmup
            lr = self.base_lr * (epoch + 1) / self.warmup_epochs
        else:
            # Cosine annealing
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))

        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

        return lr


def train_epoch(model, dataloader, optimizer, criterion, device, grad_accum_steps=1):
    """Train for one epoch with optional gradient accumulation."""
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for i, (X_batch, y_batch) in enumerate(dataloader):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        y_pred = model(X_batch)

        # Compute loss (scaled for gradient accumulation)
        loss = criterion(y_pred, y_batch) / grad_accum_steps

        # Backward pass
        loss.backward()

        # Gradient accumulation
        if (i + 1) % grad_accum_steps == 0:
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum_steps

    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)

            total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate_model(model, dataloader, scaler, device):
    """
    Evaluate model performance with multiple metrics.

    Returns:
        dict: Dictionary containing evaluation metrics
    """
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_pred = model(X_batch)

            all_preds.append(y_pred.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())

    # Concatenate all predictions and targets
    y_pred = np.concatenate(all_preds, axis=0)  # (n_samples, forecast_horizon, n_features)
    y_true = np.concatenate(all_targets, axis=0)

    # Inverse transform if scaler is provided
    if scaler is not None:
        # Reshape for inverse transform
        original_shape = y_pred.shape
        y_pred_2d = y_pred.reshape(-1, y_pred.shape[-1])
        y_true_2d = y_true.reshape(-1, y_true.shape[-1])

        y_pred_original = scaler.inverse_transform(y_pred_2d)
        y_true_original = scaler.inverse_transform(y_true_2d)

        y_pred = y_pred_original.reshape(original_shape)
        y_true = y_true_original.reshape(original_shape)

    # Calculate metrics
    mse = np.mean((y_pred - y_true) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_pred - y_true))

    # R² score
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    # MAPE (Mean Absolute Percentage Error) - avoid division by zero
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    metrics = {
        'rmse': float(rmse),
        'mae': float(mae),
        'mse': float(mse),
        'r2': float(r2),
        'mape': float(mape)
    }

    return metrics, y_pred, y_true


def plot_predictions(y_true, y_pred, n_samples=5, save_path=None):
    """Plot sample predictions vs actual values."""
    fig, axes = plt.subplots(n_samples, 1, figsize=(12, 3 * n_samples))

    if n_samples == 1:
        axes = [axes]

    for i in range(min(n_samples, len(y_true))):
        axes[i].plot(y_true[i, :, 0], label='Actual', marker='o', linestyle='-', alpha=0.7)
        axes[i].plot(y_pred[i, :, 0], label='Predicted', marker='x', linestyle='--', alpha=0.7)
        axes[i].set_xlabel('Hour')
        axes[i].set_ylabel('PM2.5')
        axes[i].set_title(f'Sample {i+1}')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Predictions plot saved to {save_path}")

    plt.close()



# ============================================================================
# Configuration
# ============================================================================
CONFIG = {
    # Data parameters
    'data_path': '/content/drive/MyDrive/combined_weather_pm25_data.csv',
    'feature_cols': [
        "pm25"
        , "temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "direct_radiation", "surface_pressure"
    ],
    'window_size': 24,  # 24 hours of historical data
    'forecast_horizon': 5,  # Predict next 24 hours
    'stride': 1,
    'test_size': 0.2,
    'random_state': 42,

    # Transformer model parameters (improved)
    'd_model': 128,  # Embedding dimension
    'nhead': 8,  # Number of attention heads (increased for better attention patterns)
    'num_encoder_layers': 3,  # Number of encoder layers
    'num_decoder_layers': 3,  # Number of decoder layers
    'dim_feedforward': 512,  # Larger feedforward dimension
    'dropout': 0.1,  # Lower dropout for transformers

    # Model architecture choice: 'encoder_decoder' or 'skip_connection'
    'model_type': 'encoder_decoder',

    # Training parameters
    'batch_size': 64,  # Larger batch size for transformer stability
    'learning_rate': 1e-4,  # Lower learning rate for transformers
    'num_epochs': 100,
    'early_stopping_patience': 10,
    'warmup_epochs': 10,  # Warmup for learning rate
    'grad_accum_steps': 1,  # Gradient accumulation steps

    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',

    # Output paths
    'output_dir': '/content/drive/MyDrive/models',
    'model_name': 'pm25_transformer_v2'
}

print("=" * 80)
print("PM2.5 Prediction Model Training (Improved Two-Layer Transformer)")
print("=" * 80)
print(f"\nDevice: {CONFIG['device']}")
if CONFIG['device'] == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Model Type: {CONFIG['model_type']}")
print()

# Create output directory
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(exist_ok=True)

# ============================================================================
# Load and prepare data
# ============================================================================
print("Loading data...")
df = pd.read_csv(CONFIG['data_path'])
print(f"Loaded {len(df)} records from {df['site'].nunique()} sites")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Columns: {df.columns.tolist()}")

# Check for missing PM2.5 values
missing_pm25 = df['pm25'].isna().sum()
print(f"\nMissing PM2.5 values: {missing_pm25} ({missing_pm25/len(df)*100:.2f}%)")

# Drop rows with missing PM2.5
if missing_pm25 > 0:
    df = df.dropna(subset=['pm25'])
    print(f"After dropping missing values: {len(df)} records")

# ============================================================================
# Create sliding windows
# ============================================================================
print("\n" + "=" * 80)
print("Creating sliding windows...")
print("=" * 80)

X, y, metadata = create_multivariate_windows(
    df,
    window_size=CONFIG['window_size'],
    forecast_horizon=CONFIG['forecast_horizon'],
    feature_cols=CONFIG['feature_cols'],
    stride=CONFIG['stride']
)

print(f"\nInput (X) shape: {X.shape}")
print(f"  - {X.shape[0]} samples")
print(f"  - {X.shape[1]} time steps")
print(f"  - {X.shape[2]} features")

print(f"\nTarget (y) shape: {y.shape}")
print(f"  - {y.shape[0]} samples")
print(f"  - {y.shape[1]} forecast horizon")
print(f"  - {y.shape[2]} features")

# ============================================================================
# Normalize data
# ============================================================================
print("\n" + "=" * 80)
print("Normalizing data...")
print("=" * 80)

print(f"PM2.5 statistics (before normalization):")
print(f"  Mean: {X[:, :, 0].mean():.2f}")
print(f"  Std: {X[:, :, 0].std():.2f}")
print(f"  Min: {X[:, :, 0].min():.2f}")
print(f"  Max: {X[:, :, 0].max():.2f}")

# ============================================================================
# Split data
# ============================================================================
print("\n" + "=" * 80)
print("Splitting data into train/test sets...")
print("=" * 80)

X_train, X_test, y_train, y_test, meta_train, meta_test = split_train_test_by_site(
    X, y, metadata,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state']
)

print(f"\nTraining set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Sites: {sorted(meta_train['site'].unique())}")

print(f"\nTest set:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")
print(f"  Sites: {sorted(meta_test['site'].unique())}")

# ============================================================================
# Normalize using training data statistics
# ============================================================================
scaler = StandardScaler()

# Reshape for scaling: (n_samples * time_steps, n_features)
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
X_test_2d = X_test.reshape(-1, X_test.shape[-1])
y_train_2d = y_train.reshape(-1, y_train.shape[-1])
y_test_2d = y_test.reshape(-1, y_test.shape[-1])

# Fit on training data
scaler.fit(X_train_2d)

# Transform both X and y
X_train_scaled = scaler.transform(X_train_2d).reshape(X_train.shape)
X_test_scaled = scaler.transform(X_test_2d).reshape(X_test.shape)
y_train_scaled = scaler.transform(y_train_2d).reshape(y_train.shape)
y_test_scaled = scaler.transform(y_test_2d).reshape(y_test.shape)

print(f"\nAfter normalization:")
print(f"  Mean: {X_train_scaled.mean():.4f}")
print(f"  Std: {X_train_scaled.std():.4f}")

# ============================================================================
# Create PyTorch datasets and dataloaders
# ============================================================================
print("\n" + "=" * 80)
print("Creating PyTorch datasets...")
print("=" * 80)

train_dataset = PM25Dataset(X_train_scaled, y_train_scaled)
test_dataset = PM25Dataset(X_test_scaled, y_test_scaled)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=0,
    pin_memory=True if CONFIG['device'] == 'cuda' else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0,
    pin_memory=True if CONFIG['device'] == 'cuda' else False
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# ============================================================================
# Initialize model
# ============================================================================
print("\n" + "=" * 80)
print(f"Initializing {CONFIG['model_type']} Transformer model...")
print("=" * 80)

device = torch.device(CONFIG['device'])

if CONFIG['model_type'] == 'encoder_decoder':
    model = PM25EncoderDecoderTransformer(
        input_size=len(CONFIG['feature_cols']),
        d_model=CONFIG['d_model'],
        nhead=CONFIG['nhead'],
        num_encoder_layers=CONFIG['num_encoder_layers'],
        num_decoder_layers=CONFIG['num_decoder_layers'],
        dim_feedforward=CONFIG['dim_feedforward'],
        dropout=CONFIG['dropout'],
        forecast_horizon=CONFIG['forecast_horizon'],
        output_size=len(CONFIG['feature_cols']),
        window_size=CONFIG['window_size']
    ).to(device)
else:
    model = PM25TransformerWithSkipConnections(
        input_size=len(CONFIG['feature_cols']),
        d_model=CONFIG['d_model'],
        nhead=CONFIG['nhead'],
        num_layers_per_block=CONFIG['num_encoder_layers'],
        dim_feedforward=CONFIG['dim_feedforward'],
        dropout=CONFIG['dropout'],
        forecast_horizon=CONFIG['forecast_horizon'],
        output_size=len(CONFIG['feature_cols']),
        window_size=CONFIG['window_size']
    ).to(device)

print(f"\nModel architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")



Mounted at /content/drive
CUDA is available. Using GPU.
Device set to: cuda
PM2.5 Prediction Model Training (Improved Two-Layer Transformer)

Device: cuda
GPU: Tesla T4
Model Type: encoder_decoder

Loading data...
Loaded 1743253 records from 87 sites
Date range: 2013-02-24 to 2025-11-12
Columns: ['time', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'precipitation', 'wind_speed_10m', 'wind_direction_10m', 'direct_radiation_instant', 'direct_radiation', 'pressure_msl', 'surface_pressure', 'cloud_cover', 'incident_id', 'site', 'pm25', 'date', 'start_hour']

Missing PM2.5 values: 0 (0.00%)

Creating sliding windows...
Processing 87 sites to create sliding windows...


Sites:   0%|          | 0/87 [00:00<?, ?it/s]


Input (X) shape: (1740821, 24, 7)
  - 1740821 samples
  - 24 time steps
  - 7 features

Target (y) shape: (1740821, 5, 7)
  - 1740821 samples
  - 5 forecast horizon
  - 7 features

Normalizing data...
PM2.5 statistics (before normalization):
  Mean: 15.01
  Std: 25.85
  Min: -13.70
  Max: 1248.90

Splitting data into train/test sets...

Training set:
  X_train shape: (1557223, 24, 7)
  y_train shape: (1557223, 5, 7)
  Sites: [np.int64(2032), np.int64(2094), np.int64(2105), np.int64(2143), np.int64(2208), np.int64(2224), np.int64(2263), np.int64(2266), np.int64(2320), np.int64(2353), np.int64(2360), np.int64(2410), np.int64(2420), np.int64(2460), np.int64(2485), np.int64(2499), np.int64(2551), np.int64(2593), np.int64(2596), np.int64(2622), np.int64(2628), np.int64(2630), np.int64(2655), np.int64(2744), np.int64(2752), np.int64(2829), np.int64(2831), np.int64(2849), np.int64(2878), np.int64(2880), np.int64(2899), np.int64(2915), np.int64(2943), np.int64(2956), np.int64(2958), np.int64(

/tmp/ipykernel_1212/833660265.py:319: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(



Model architecture:
PM25EncoderDecoderTransformer(
  (encoder_embedding): TemporalEmbedding(
    (input_projection): Linear(in_features=7, out_features=128, bias=True)
    (pos_encoding): LearnablePositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout

In [2]:
# @title
# ============================================================================
# Training setup
# ============================================================================
criterion = nn.MSELoss()

# Use AdamW optimizer (better for transformers)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=0.01,
    betas=(0.9, 0.98)
)

# Warmup cosine scheduler
scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs=CONFIG['warmup_epochs'],
    total_epochs=CONFIG['num_epochs']
)

early_stopping = EarlyStopping(patience=CONFIG['early_stopping_patience'])

# ============================================================================
# Training loop
# ============================================================================
print("\n" + "=" * 80)
print("Training model...")
print("=" * 80)

train_losses = []
val_losses = []
learning_rates = []
best_val_loss = float('inf')

for epoch in range(CONFIG['num_epochs']):
    # Update learning rate
    current_lr = scheduler.step(epoch)
    learning_rates.append(current_lr)

    # Train
    train_loss = train_epoch(
        model, train_loader, optimizer, criterion, device,
        grad_accum_steps=CONFIG['grad_accum_steps']
    )
    train_losses.append(train_loss)

    # Validate
    val_loss = validate(model, test_loader, criterion, device)
    val_losses.append(val_loss)

    # Print progress
    print(f"Epoch [{epoch+1}/{CONFIG['num_epochs']}] "
          f"LR: {current_lr:.6f} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'config': CONFIG
        }, output_dir / f"{CONFIG['model_name']}_best.pth")
        print(f"  → Saved best model (val_loss: {val_loss:.6f})")

    # Early stopping
    if early_stopping(val_loss, model):
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        # Restore best model
        model.load_state_dict(early_stopping.best_model_state)
        break

# ============================================================================
# Plot training history
# ============================================================================
print("\n" + "=" * 80)
print("Plotting training history...")
print("=" * 80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(train_losses, label='Training Loss', alpha=0.7)
ax1.plot(val_losses, label='Validation Loss', alpha=0.7)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.set_title('Training History (Two-Layer Transformer)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Learning rate plot
ax2.plot(learning_rates, label='Learning Rate', color='green', alpha=0.7)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule (Warmup + Cosine)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / f"{CONFIG['model_name']}_training_history.png", dpi=300)
print(f"Training history saved to {output_dir / f'{CONFIG['model_name']}_training_history.png'}")
plt.close()

# ============================================================================
# Evaluate model
# ============================================================================
print("\n" + "=" * 80)
print("Evaluating model...")
print("=" * 80)

# Load best model
checkpoint = torch.load(output_dir / f"{CONFIG['model_name']}_best.pth")
model.load_state_dict(checkpoint['model_state_dict'])

# Evaluate on test set
test_metrics, y_pred, y_true = evaluate_model(model, test_loader, scaler, device)

print("\nTest Set Performance:")
print(f"  RMSE: {test_metrics['rmse']:.4f}")
print(f"  MAE:  {test_metrics['mae']:.4f}")
print(f"  MSE:  {test_metrics['mse']:.4f}")
print(f"  R²:   {test_metrics['r2']:.4f}")
print(f"  MAPE: {test_metrics['mape']:.2f}%")

# Evaluate on training set for comparison
train_metrics, _, _ = evaluate_model(model, train_loader, scaler, device)

print("\nTraining Set Performance:")
print(f"  RMSE: {train_metrics['rmse']:.4f}")
print(f"  MAE:  {train_metrics['mae']:.4f}")
print(f"  R²:   {train_metrics['r2']:.4f}")

# ============================================================================
# Plot sample predictions
# ============================================================================
print("\n" + "=" * 80)
print("Plotting sample predictions...")
print("=" * 80)

plot_predictions(
    y_true, y_pred,
    n_samples=5,
    save_path=output_dir / f"{CONFIG['model_name']}_predictions.png"
)

# ============================================================================
# Save scaler and final artifacts
# ============================================================================
print("\n" + "=" * 80)
print("Saving artifacts...")
print("=" * 80)

# Save scaler
import joblib
joblib.dump(scaler, output_dir / f"{CONFIG['model_name']}_scaler.pkl")
print(f"Scaler saved to {output_dir / f'{CONFIG['model_name']}_scaler.pkl'}")

# Save configuration and metrics
results = {
    'config': CONFIG,
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'training_history': {
        'train_losses': [float(x) for x in train_losses],
        'val_losses': [float(x) for x in val_losses],
        'learning_rates': [float(x) for x in learning_rates]
    },
    'timestamp': datetime.now().isoformat()
}

with open(output_dir / f"{CONFIG['model_name']}_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {output_dir / f'{CONFIG['model_name']}_results.json'}")

print("\n" + "=" * 80)
print("Training completed successfully!")
print("=" * 80)
print(f"\nModel files saved in: {output_dir}")
print(f"  - {CONFIG['model_name']}_best.pth (model checkpoint)")
print(f"  - {CONFIG['model_name']}_scaler.pkl (data scaler)")
print(f"  - {CONFIG['model_name']}_results.json (metrics and config)")
print(f"  - {CONFIG['model_name']}_training_history.png (loss curves)")
print(f"  - {CONFIG['model_name']}_predictions.png (sample predictions)")


Training model...
Epoch [1/100] LR: 0.000010 | Train Loss: 0.236978 | Val Loss: 0.140874
  → Saved best model (val_loss: 0.140874)
Epoch [2/100] LR: 0.000020 | Train Loss: 0.142572 | Val Loss: 0.116390
  → Saved best model (val_loss: 0.116390)
Epoch [3/100] LR: 0.000030 | Train Loss: 0.125471 | Val Loss: 0.107738
  → Saved best model (val_loss: 0.107738)
Epoch [4/100] LR: 0.000040 | Train Loss: 0.118440 | Val Loss: 0.105115
  → Saved best model (val_loss: 0.105115)
Epoch [5/100] LR: 0.000050 | Train Loss: 0.113978 | Val Loss: 0.102976
  → Saved best model (val_loss: 0.102976)
Epoch [6/100] LR: 0.000060 | Train Loss: 0.110797 | Val Loss: 0.102694
  → Saved best model (val_loss: 0.102694)
Epoch [7/100] LR: 0.000070 | Train Loss: 0.108582 | Val Loss: 0.102319
  → Saved best model (val_loss: 0.102319)
Epoch [8/100] LR: 0.000080 | Train Loss: 0.106855 | Val Loss: 0.100598
  → Saved best model (val_loss: 0.100598)
Epoch [9/100] LR: 0.000090 | Train Loss: 0.105392 | Val Loss: 0.101945
EarlyS

KeyboardInterrupt: 

In [ ]:
import torch
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Re-import the Transformer model classes (they should already be in your notebook)
# If not, you'll need to copy them from the training notebook

# Configuration (should match your training setup)
MODEL_DIR = Path('/content/drive/MyDrive/models/pm25_only_transformer_24_steps')
MODEL_NAME = 'pm25_transformer_v2'

print("Loading trained Transformer model and scaler...")

# Load scaler
scaler = joblib.load(MODEL_DIR / f"{MODEL_NAME}_scaler.pkl")

# Load model checkpoint FIRST to get the correct CONFIG
checkpoint = torch.load(MODEL_DIR / f"{MODEL_NAME}_best.pth", map_location=device)

# Use CONFIG from checkpoint (this is the actual config used during training)
CONFIG = checkpoint['config']

print(f"Model configuration:")
print(f"  Model type: {CONFIG['model_type']}")
print(f"  Forecast horizon: {CONFIG['forecast_horizon']} hours")
print(f"  Window size: {CONFIG['window_size']} hours")
print(f"  d_model: {CONFIG['d_model']}")
print(f"  nhead: {CONFIG['nhead']}")

# Initialize model with the CORRECT config from checkpoint
if CONFIG['model_type'] == 'encoder_decoder':
    model = PM25EncoderDecoderTransformer(
        input_size=len(CONFIG['feature_cols']),
        d_model=CONFIG['d_model'],
        nhead=CONFIG['nhead'],
        num_encoder_layers=CONFIG['num_encoder_layers'],
        num_decoder_layers=CONFIG['num_decoder_layers'],
        dim_feedforward=CONFIG['dim_feedforward'],
        dropout=CONFIG['dropout'],
        forecast_horizon=CONFIG['forecast_horizon'],
        output_size=len(CONFIG['feature_cols']),
        window_size=CONFIG['window_size']
    ).to(device)
else:  # skip_connection
    model = PM25TransformerWithSkipConnections(
        input_size=len(CONFIG['feature_cols']),
        d_model=CONFIG['d_model'],
        nhead=CONFIG['nhead'],
        num_layers_per_block=CONFIG['num_encoder_layers'],
        dim_feedforward=CONFIG['dim_feedforward'],
        dropout=CONFIG['dropout'],
        forecast_horizon=CONFIG['forecast_horizon'],
        output_size=len(CONFIG['feature_cols']),
        window_size=CONFIG['window_size']
    ).to(device)

# Load trained weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\n✓ Model loaded successfully (val_loss: {checkpoint['val_loss']:.6f})")
print(f"✓ Scaler loaded")

# ============================================================================
# CELL 2: Select Continuous Week from Test Data (Containing Max PM2.5)
# ============================================================================

print("Selecting continuous week containing maximum PM2.5 event...")

# Parameters
HOURS_PER_DAY = 24
PREDICTION_DAYS = 7
TOTAL_HOURS = HOURS_PER_DAY * PREDICTION_DAYS  # 168 hours

# Get test data (reuse the test sites from training split)
test_df = df[df['site'].isin(meta_test['site'].unique())].copy()
test_df = test_df.sort_values(['site', 'date', 'start_hour']).reset_index(drop=True)

# Find a site and window that contains the maximum PM2.5 value
selected_site = None
selected_start_idx = None
max_pm25_value = None

for site in meta_test['site'].unique():
    site_data = test_df[test_df['site'] == site].reset_index(drop=True)

    # Need enough data for initial window + 7 days of predictions
    required_hours = CONFIG['window_size'] + TOTAL_HOURS

    if len(site_data) < required_hours:
        continue

    # Find the maximum PM2.5 value for this site
    max_pm25_idx = site_data['pm25'].idxmax()
    site_max_pm25 = site_data.loc[max_pm25_idx, 'pm25']

    # Check if we can create a valid 7-day window that includes this max value
    # The max should be in the 7-day prediction window (not the initial input window)
    # So: start_idx + window_size <= max_idx < start_idx + window_size + TOTAL_HOURS

    # Calculate valid start indices
    earliest_start = max(0, max_pm25_idx - CONFIG['window_size'] - TOTAL_HOURS + 1)
    latest_start = max_pm25_idx - CONFIG['window_size']

    # Make sure we have enough data after the start index
    for start_idx in range(earliest_start, latest_start + 1):
        if start_idx >= 0 and start_idx + required_hours <= len(site_data):
            selected_site = site
            selected_start_idx = start_idx
            max_pm25_value = site_max_pm25
            break

    if selected_site is not None:
        break

if selected_site is None:
    # Fallback: just find any site with enough data
    print("Warning: Could not find window containing max PM2.5. Using fallback selection...")
    for site in meta_test['site'].unique():
        site_data = test_df[test_df['site'] == site].reset_index(drop=True)
        required_hours = CONFIG['window_size'] + TOTAL_HOURS

        if len(site_data) >= required_hours:
            selected_start_idx = len(site_data) // 4
            selected_site = site
            max_pm25_value = site_data['pm25'].max()
            break

if selected_site is None:
    raise ValueError("No test site has enough continuous data")

# Extract the week's data
site_data = test_df[test_df['site'] == selected_site].reset_index(drop=True)
end_idx = selected_start_idx + CONFIG['window_size'] + TOTAL_HOURS
week_data = site_data.iloc[selected_start_idx:end_idx].copy()

# Verify max PM2.5 is in the prediction window
prediction_window = week_data.iloc[CONFIG['window_size']:].copy()
max_in_window = prediction_window['pm25'].max()

print(f"Selected site: {selected_site}")
print(f"Date range: {week_data['date'].min()} to {week_data['date'].max()}")
print(f"Total hours: {len(week_data)}")
print(f"PM2.5 range: {week_data['pm25'].min():.2f} - {week_data['pm25'].max():.2f}")
print(f"Max PM2.5 in prediction window: {max_in_window:.2f} μg/m³")
print(f"Site's overall max PM2.5: {max_pm25_value:.2f} μg/m³")
print(f"✓ Window contains {'MAXIMUM' if abs(max_in_window - max_pm25_value) < 0.01 else 'HIGH'} PM2.5 event")

# ============================================================================
# CELL 3: Define Prediction Function
# ============================================================================

def predict_next_hours(model, input_window, scaler, device, n_hours=24):
    """
    Predict next n hours using the Transformer model.

    Args:
        model: Trained PyTorch Transformer model
        input_window: Input data of shape (window_size, n_features)
        scaler: Fitted StandardScaler
        device: torch device
        n_hours: Number of hours to predict

    Returns:
        predictions: Array of shape (n_hours, n_features)
    """
    # Normalize input
    input_scaled = scaler.transform(input_window)

    # Convert to tensor and add batch dimension
    input_tensor = torch.FloatTensor(input_scaled).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(input_tensor)  # (1, forecast_horizon, n_features)

    # Convert back to numpy
    output_scaled = output.cpu().numpy()[0]  # (forecast_horizon, n_features)

    # Take only n_hours if forecast_horizon > n_hours
    output_scaled = output_scaled[:n_hours, :]

    # Inverse transform to original scale
    predictions = scaler.inverse_transform(output_scaled)

    return predictions

# ============================================================================
# CELL 4: Generate Iterative Predictions
# ============================================================================

print("Generating week-long predictions with Transformer...")
print("=" * 60)

# Storage for results
actual_values = []
predicted_values = []
prediction_dates = []
prediction_hours = []
day_labels = []

# Extract feature data
feature_data = week_data[CONFIG['feature_cols']].values

# Start with Day 1 as initial window
current_idx = 0

# Store Day 1 (input window, no predictions)
print(f"Day 1 (Hours 0-{CONFIG['window_size']-1}): Input window")
for i in range(CONFIG['window_size']):
    actual_values.append(feature_data[i])
    predicted_values.append(np.array([np.nan]))  # No prediction for Day 1
    prediction_dates.append(week_data.iloc[i]['date'])
    prediction_hours.append(week_data.iloc[i]['start_hour'])
    day_labels.append(1)

# Initial window
current_window = feature_data[current_idx:current_idx + CONFIG['window_size']]

# Predict Days 2-7 iteratively
for day in range(2, PREDICTION_DAYS + 1):
    # Predict next hours (limited by forecast_horizon)
    predictions = predict_next_hours(
        model,
        current_window,
        scaler,
        device,
        n_hours=min(HOURS_PER_DAY, CONFIG['forecast_horizon'])
    )

    # Get actual values for comparison
    actual_start_idx = current_idx + CONFIG['window_size']
    actual_end_idx = actual_start_idx + len(predictions)
    actual_day = feature_data[actual_start_idx:actual_end_idx]

    # Calculate metrics
    mae = np.mean(np.abs(predictions[:, 0] - actual_day[:, 0]))
    rmse = np.sqrt(np.mean((predictions[:, 0] - actual_day[:, 0]) ** 2))

    print(f"Day {day} (Hours {actual_start_idx}-{actual_end_idx-1}) - {len(predictions)}hr forecast:")
    print(f"  Actual:    {actual_day[:, 0].mean():.2f} ± {actual_day[:, 0].std():.2f} μg/m³")
    print(f"  Predicted: {predictions[:, 0].mean():.2f} ± {predictions[:, 0].std():.2f} μg/m³")
    print(f"  MAE: {mae:.4f}, RMSE: {rmse:.4f}")

    # Store results for the predicted hours
    for i in range(len(predictions)):
        idx = actual_start_idx + i
        actual_values.append(feature_data[idx])
        predicted_values.append(predictions[i])
        prediction_dates.append(week_data.iloc[idx]['date'])
        prediction_hours.append(week_data.iloc[idx]['start_hour'])
        day_labels.append(day)

    # If forecast horizon was less than 24 hours, fill remaining hours with NaN
    remaining_hours = HOURS_PER_DAY - len(predictions)
    if remaining_hours > 0:
        for i in range(remaining_hours):
            idx = actual_end_idx + i
            actual_values.append(feature_data[idx])
            predicted_values.append(np.array([np.nan]))
            prediction_dates.append(week_data.iloc[idx]['date'])
            prediction_hours.append(week_data.iloc[idx]['start_hour'])
            day_labels.append(day)

    # Slide window by 24 hours for next iteration (using actual data)
    current_idx += HOURS_PER_DAY
    current_window = feature_data[current_idx:current_idx + CONFIG['window_size']]

print("=" * 60)
print(f"Note: Transformer forecast horizon is {CONFIG['forecast_horizon']} hours")
if CONFIG['forecast_horizon'] < HOURS_PER_DAY:
    print(f"  → Each day shows {CONFIG['forecast_horizon']}hr predictions + {HOURS_PER_DAY - CONFIG['forecast_horizon']}hr no prediction")

# ============================================================================
# CELL 5: Create Results DataFrame
# ============================================================================

# Create DataFrame
results_df = pd.DataFrame({
    'day': day_labels,
    'date': prediction_dates,
    'hour': prediction_hours,
    'actual_pm25': [v[0] for v in actual_values],
    'predicted_pm25': [v[0] if len(v) > 0 and not np.isnan(v[0]) else np.nan for v in predicted_values],
    'site': selected_site
})

# Calculate errors (will be NaN where prediction is NaN)
results_df['prediction_error'] = results_df['predicted_pm25'] - results_df['actual_pm25']
results_df['absolute_error'] = np.abs(results_df['prediction_error'])

# Display sample
print("\nResults DataFrame (first 30 hours):")
print(results_df.head(30))

# Overall statistics (only for hours with predictions)
predicted_days = results_df[(results_df['day'] >= 2) & (~results_df['predicted_pm25'].isna())]

if len(predicted_days) > 0:
    overall_mae = predicted_days['absolute_error'].mean()
    overall_rmse = np.sqrt(np.mean(predicted_days['prediction_error']**2))

    print(f"\n{'='*60}")
    print(f"OVERALL PERFORMANCE (Days 2-7, {len(predicted_days)} hours predicted):")
    print(f"{'='*60}")
    print(f"MAE:  {overall_mae:.4f} μg/m³")
    print(f"RMSE: {overall_rmse:.4f} μg/m³")

    if CONFIG['forecast_horizon'] < HOURS_PER_DAY:
        missing_hours = len(results_df[results_df['day'] >= 2]) - len(predicted_days)
        print(f"\nNote: {missing_hours} hours had no predictions (forecast horizon = {CONFIG['forecast_horizon']}hr)")

    print(f"{'='*60}")
else:
    print("\nNo predictions available for overall metrics")

Loading trained Transformer model and scaler...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/models/pm25_only_transformer_24_steps/pm25_transformer_v2_scaler.pkl'

In [ ]:
results_df.to_csv(MODEL_DIR / f"{MODEL_NAME}_predictions.csv", index=False)